In [ ]:
import importlib, anonlab
import anonlab.anonymizer as _an
import anonlab.attacker as _att
import anonlab.attacker_module as _am

importlib.reload(anonlab); importlib.reload(_an); importlib.reload(_att); importlib.reload(_am)

from anonlab import make_synthetic_patients, anonymize_qi, group_size_summary, make_attacker_subset_and_validate
print("OK imports:", callable(make_synthetic_patients), callable(anonymize_qi), callable(make_attacker_subset_and_validate))

# minimal end-to-end smoke test (no glucose)
df = make_synthetic_patients(n=1000, seed=421) 


OK imports: True True True


In [28]:
score=[]

for k in [0,1,2,3,4,5,10,15]:
    
    a = anonymize_qi(df, k=k, age_bin_width=0, topcode_start=None, rare_zip_min_frac=0.00, max_iter=0, extra_iter=0, allow_suppression=True)

    att = make_attacker_subset_and_validate(df, a, fraction=0.1, seed=1)
    attacker_orig = att["attacker_orig"][["person_id","age","zip3","sex"]]
    attacker_ids  = att["attacker_ids"]

    res = _am.run_attack(attacker_orig, a, attacker_ids, solver=("ortools" if getattr(_am,"ORTOOLS_AVAILABLE",False) else "greedy"))
    print(f"k={k} Hit@1: {res['eval']['hits']}/{res['eval']['n_attack']} rate={res['eval']['hit_rate']:.3f}")
    score.append((k, res['eval']['hit_rate']))

k=0 Hit@1: 54/100 rate=0.540
k=1 Hit@1: 54/100 rate=0.540
k=2 Hit@1: 26/100 rate=0.260
k=3 Hit@1: 10/100 rate=0.100
k=4 Hit@1: 3/100 rate=0.030
k=5 Hit@1: 1/100 rate=0.010
k=10 Hit@1: 0/100 rate=0.000
k=15 Hit@1: 0/100 rate=0.000


In [30]:
from anonlab import attacker_module as _am

print("ORTOOLS_AVAILABLE inside module =", getattr(_am, "ORTOOLS_AVAILABLE", None))


ORTOOLS_AVAILABLE inside module = True
